## Pretraining

### Load all allowed libraries

In [1]:
import numpy as np
from PIL import Image
import pandas as pd
import sklearn
import scipy
import seaborn as sns
import torch
import torchinfo
import torchvision
from tqdm import tqdm

import matplotlib.pyplot as plt

Could not save font_manager cache [Errno 28] No space left on device


### Define data transfromation function, and data augmentation

In [15]:
from torchvision.transforms import v2

# Training data transforms and augmentation
training_tfms_and_agmt = v2.Compose([
    v2.ToImage(), # this turns it into a tensor

    v2.RandomRotation(degrees=10), # for realistic pet location variations 
    v2.RandomAffine(degrees=0, translate=(0.1, 0.1)), # for better position accuracy
    v2.RandomHorizontalFlip(p=0.5), # it is still the same cat if flip like this but double data

    v2.RandomResizedCrop((224, 224), scale=(0.8, 1.0), antialias=True), # 224 * 224 seems to be what everyone doing
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05), # this is for pet photograph varience

    v2.ToDtype(torch.float32, scale=True), # apply the min-max normalization and scale 0-255 to num between 0 and 1
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), # apply the standardization for the z-scores things
])

valutation_tfms = v2.Compose([
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

### Load dataset and create custom dataset

In [ ]:
from torch.utils.data import Dataset, random_split

class TransformWrapper(Dataset):
    def __init__(self, subset, transform=None):
        super().__init__() # inherit the Dataset init method just in case even though I think there is nothing
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

raw_data = datasets.OxfordIIITPet(root="/shared/storage/cs/studentscratch/cqh514", split='trainval', download=True)


### Randomly split but make the split constant accross different runes, in a 80:20 training, valuation configuration

In [ ]:
# do the dataset splitting
dataset_size = len(raw_data)
num_training = int(0.8 * dataset_size)
num_valuation = dataset_size - num_training

# define the manual seed so it no change after training
generator = torch.Generator().manual_seed(67)
raw_training, raw_valuation = random_split(raw_data, [num_training, num_valuation], generator=generator)

### Inject the transforms to the raw subset that is created with the split we just made

In [22]:
# Wrap the raw subsets to inject the transforms on the fly
training_dataset = TransformWrapper(raw_training, transform=training_tfms_and_agmt)
valuation_dataset = TransformWrapper(raw_valuation, transform=valutation_tfms)

### Preparing data with dataloaders, don't shuffle the valuation then no waste cpu

In [23]:
from torch.utils.data import DataLoader
training_loader = DataLoader(training_dataset, batch_size=32, shuffle=True, num_workers=4)
valuation_loader = DataLoader(valuation_dataset, batch_size=32, shuffle=False, num_workers=4)

### Define the first 1080ti in the server for training

In [25]:
device = torch.device("cuda")

# Training

### Define the neural network by subclassing the nn.Module

In [ ]:
import torch.nn as nn


